In [ ]:
!pip install faker xgboost lightgbm shap scikit-learn pandas numpy matplotlib seaborn -q

In [ ]:
import pandas as pd
import numpy as np
from faker import Faker
from collections import defaultdict
import random
import networkx as nx

In [ ]:
#--------------------------
# Config
# --------------------------
N_APPLICANTS = 4000
FRAUD_RATIO = 0.18
RANDOM_STATE = 42
AVG_TXNS_PER_ACCOUNT = 12

np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
fake = Faker('en_IN')
Faker.seed(RANDOM_STATE)

In [ ]:
# ===============================================================
# 1. Generate Applicants
# ===============================================================
def generate_applicants(n_applicants=4000, fraud_ratio=0.18):
    data = []
    shared_devices = [fake.uuid4() for _ in range(120)]
    shared_ips = [fake.ipv4() for _ in range(90)]
    shared_phones = [fake.phone_number() for _ in range(100)]

    for i in range(n_applicants):
        is_fraud = 1 if np.random.rand() < fraud_ratio else 0
        age = np.random.randint(21, 55)

        if is_fraud:
            difficulty = np.random.choice(['easy', 'medium', 'hard'], p=[0.3, 0.3, 0.4])
            if difficulty == 'easy':
                work_exp = max(0, age - np.random.randint(8, 14))
                job = np.random.choice(['Student', 'Unemployed'])
                income = np.random.randint(12, 35) * 100000
                device = np.random.choice(shared_devices)
                ip = np.random.choice(shared_ips)
                phone = np.random.choice(shared_phones)
                address_changes = np.random.randint(3, 7)
                num_apps = np.random.randint(3, 7)
            elif difficulty == 'medium':
                work_exp = max(0, age - np.random.randint(15, 21))
                job = np.random.choice(['Salaried', 'Self-employed', 'Student'])
                income = np.random.randint(8, 25) * 100000
                device = np.random.choice(shared_devices) if np.random.rand() < 0.5 else fake.uuid4()
                ip = np.random.choice(shared_ips) if np.random.rand() < 0.4 else fake.ipv4()
                phone = fake.phone_number()
                address_changes = np.random.randint(2, 5)
                num_apps = np.random.randint(2, 5)
            else:
                work_exp = max(0, age - np.random.randint(18, 24))
                job = np.random.choice(['Salaried', 'Self-employed'])
                income = np.random.randint(5, 18) * 100000
                device = np.random.choice(shared_devices) if np.random.rand() < 0.25 else fake.uuid4()
                ip = fake.ipv4()
                phone = fake.phone_number()
                address_changes = np.random.randint(0, 3)
                num_apps = np.random.randint(1, 3)
            fraud_difficulty = difficulty
        else:
            work_exp = max(0, age - np.random.randint(19, 26))
            job = np.random.choice(['Salaried', 'Self-employed', 'Government', 'Student'], p=[0.55, 0.25, 0.1, 0.1])
            income = np.random.randint(2, 22) * 100000
            if job == 'Student':
                income = np.random.randint(0, 5) * 100000
            device = fake.uuid4()
            ip = fake.ipv4()
            phone = fake.phone_number()
            address_changes = np.random.randint(0, 3)
            num_apps = np.random.randint(1, 3)
            fraud_difficulty = 'Genuine'

        data.append({
            'applicant_id': f'APP{100000 + i}',
            'age': age,
            'work_experience': work_exp,
            'declared_income': income,
            'job_category': job,
            'device_id': device,
            'ip_address': ip,
            'phone_number': phone,
            'address_change_count': address_changes,
            'num_applications': num_apps,
            'fraud_difficulty': fraud_difficulty,
            'label': is_fraud
        })

    return pd.DataFrame(data)

applicants = generate_applicants(N_APPLICANTS, FRAUD_RATIO)
print("Applicants shape:", applicants.shape)
print(applicants['label'].value_counts(normalize=True))

Applicants shape: (4000, 12)
label
0    0.82925
1    0.17075
Name: proportion, dtype: float64


In [ ]:
# STEP 2: Diagnose why ICS equals 1.0

components = [
    'demographic_coherence',
    'relational_coherence',
    'behavioral_coherence',
    'identity_coherence_score'
]

print("Mean component scores by label:")
print(applicants.groupby('label')[components].mean().round(4))

print("\nPercentage of applicants with perfect component scores:")
for col in components:
    pct = (applicants[col] >= 0.999999).mean() * 100
    print(f"{col}: {pct:.2f}%")

print("\nApplicants with ICS = 1.0, by label:")
perfect = applicants[applicants['identity_coherence_score'] >= 0.999999]
print(perfect['label'].value_counts().sort_index())
print(
    perfect['label'].value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nDemographic rule overlap:")
print(pd.crosstab(
    applicants['rule_age_exp'],
    (applicants['age_exp_gap'] < 10).astype(int),
    rownames=['rule_age_exp'],
    colnames=['age_exp_gap < 10']
))

Mean component scores by label:
       demographic_coherence  relational_coherence  behavioral_coherence  \
label                                                                      
0                     0.9945                1.0000                 1.000   
1                     0.6868                0.6862                 0.598   

       identity_coherence_score  
label                            
0                        0.9978  
1                        0.6644  

Percentage of applicants with perfect component scores:
demographic_coherence: 87.30%
relational_coherence: 89.90%
behavioral_coherence: 90.95%
identity_coherence_score: 83.67%

Applicants with ICS = 1.0, by label:
label
0    3135
1     212
Name: count, dtype: int64
label
0    93.67
1     6.33
Name: proportion, dtype: float64

Demographic rule overlap:
age_exp_gap < 10     0   1
rule_age_exp              
0                 3723   0
1                  203  74


In [ ]:
# Diagnose the relational penalty and sharing features

share_cols = [
    'device_share',
    'ip_share',
    'phone_share',
    'device_share_log',
    'ip_share_log',
    'phone_share_log'
]

print("Sharing features by label:")
print(applicants.groupby('label')[share_cols].mean().round(3))

print("\nRelational coherence by label:")
print(
    applicants.groupby('label')['relational_coherence']
    .agg(['mean', 'std', 'min', 'max'])
    .round(4)
)

print("\nRelational coherence value counts:")
print(
    applicants['relational_coherence']
    .round(3)
    .value_counts()
    .sort_index()
    .head(20)
)

print("\nPerfect relational coherence by label (%):")
print(
    applicants.assign(
        perfect_rel=applicants['relational_coherence'] >= 0.999999
    )
    .groupby('label')['perfect_rel']
    .mean()
    .mul(100)
    .round(2)
)

Sharing features by label:
       device_share  ip_share  phone_share  device_share_log  ip_share_log  \
label                                                                        
0             1.000     1.000        1.000             0.693         0.693   
1             2.534     2.215        1.583             1.151         1.048   

       phone_share_log  
label                   
0                0.693  
1                0.876  

Relational coherence by label:
         mean     std  min  max
label                          
0      1.0000  0.0000  1.0  1.0
1      0.6862  0.3112  0.0  1.0

Relational coherence value counts:
relational_coherence
0.000    1
0.034    1
0.047    1
0.051    1
0.054    1
0.065    1
0.069    1
0.073    1
0.085    4
0.093    1
0.103    4
0.104    1
0.111    2
0.119    1
0.123    2
0.125    1
0.142    4
0.146    1
0.150    7
0.151    1
Name: count, dtype: int64

Perfect relational coherence by label (%):
label
0    100.00
1     40.85
Name: perfect_rel, dtyp

In [ ]:
# ===============================================================
# 2. Generate Accounts (linked to Applicants)
# ===============================================================
def generate_accounts(applicants_df):
    accounts = []
    account_counter = 200000

    for _, row in applicants_df.iterrows():
        # Genuine → mostly 1 account, Fraud → sometimes 2-3 accounts
        if row['label'] == 1:
            n_accounts = np.random.choice([1, 2, 3], p=[0.5, 0.35, 0.15])
        else:
            n_accounts = np.random.choice([1, 2], p=[0.85, 0.15])

        for _ in range(n_accounts):
            accounts.append({
                'account_id': f'ACC{account_counter}',
                'applicant_id': row['applicant_id'],
                'label': row['label']          # account label = applicant label
            })
            account_counter += 1

    return pd.DataFrame(accounts)

accounts = generate_accounts(applicants)
print("\nAccounts shape:", accounts.shape)
print("Accounts per applicant (mean):", accounts.groupby('applicant_id').size().mean())


Accounts shape: (4906, 3)
Accounts per applicant (mean): 1.2265


In [ ]:
# ===============================================================
# 3. Generate Transactions (with Fraud Rings)
# ===============================================================
def generate_transactions(accounts_df, avg_txns=12):
    transactions = []
    account_ids = accounts_df['account_id'].tolist()
    label_map = dict(zip(accounts_df['account_id'], accounts_df['label']))

    fraud_accounts = accounts_df[accounts_df['label'] == 1]['account_id'].tolist()
    genuine_accounts = accounts_df[accounts_df['label'] == 0]['account_id'].tolist()

    txn_id = 300000

    # Normal transactions (mostly genuine)
    for acc in account_ids:
        n_txns = np.random.randint(avg_txns - 4, avg_txns + 5)
        for _ in range(n_txns):
            if label_map[acc] == 1 and np.random.rand() < 0.55:
                # Fraud account prefers other fraud accounts
                receiver = np.random.choice(fraud_accounts)
            else:
                receiver = np.random.choice(account_ids)

            if receiver == acc:
                continue

            amount = np.random.randint(500, 75000)
            transactions.append({
                'txn_id': f'TXN{txn_id}',
                'sender_account': acc,
                'receiver_account': receiver,
                'amount': amount
            })
            txn_id += 1

    return pd.DataFrame(transactions)

transactions = generate_transactions(accounts, AVG_TXNS_PER_ACCOUNT)
print("\nTransactions shape:", transactions.shape)


Transactions shape: (59055, 4)


In [ ]:
# ===============================================================
# 4. Save all files
# ===============================================================
applicants.to_csv('linked_applicants.csv', index=False)
accounts.to_csv('linked_accounts.csv', index=False)
transactions.to_csv('linked_transactions.csv', index=False)

print("\nFiles saved:")
print(" - linked_applicants.csv")
print(" - linked_accounts.csv")
print(" - linked_transactions.csv")
print("\nLinked Synthetic Dataset Generation Completed!")


Files saved:
 - linked_applicants.csv
 - linked_accounts.csv
 - linked_transactions.csv

Linked Synthetic Dataset Generation Completed!


In [ ]:
# Check whether account creation itself reveals the label

accounts_per_applicant = (
    accounts.groupby("applicant_id")
    .agg(
        label=("label", "first"),
        account_count=("account_id", "count")
    )
)

print("Accounts per applicant by label:")
print(
    accounts_per_applicant.groupby("label")["account_count"]
    .agg(["count", "mean", "min", "max"])
)

# Check how often transactions connect two fraud accounts
label_map = accounts.set_index("account_id")["label"]

tx = transactions.copy()
tx["sender_label"] = tx["sender_account"].map(label_map)
tx["receiver_label"] = tx["receiver_account"].map(label_map)

print("\nTransaction label combinations:")
print(pd.crosstab(tx["sender_label"], tx["receiver_label"], normalize="index").round(3))

Accounts per applicant by label:
       count      mean  min  max
label                           
0       3317  1.137775    1    2
1        683  1.657394    1    3

Transaction label combinations:
receiver_label      0      1
sender_label                
0               0.769  0.231
1               0.347  0.653


In [ ]:
# ===============================================================
# STEP 1: Load Data + Compute Identity Coherence Score
# ===============================================================

import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler, LabelEncoder

applicants = pd.read_csv('linked_applicants.csv')
accounts = pd.read_csv('linked_accounts.csv')
transactions = pd.read_csv('linked_transactions.csv')

print("Applicants:", applicants.shape)
print("Accounts  :", accounts.shape)
print("Transactions:", transactions.shape)

# --------------------------
# Apply Rules + Basic Features on Applicants
# --------------------------
def apply_rules_and_features(df):
    df = df.copy()

    # Rules
    df['rule_age_exp'] = ((df['work_experience'] > (df['age'] - 17)) |
                          ((df['age'] <= 23) & (df['work_experience'] >= 5))).astype(int)
    df['rule_income_job'] = (((df['job_category'].isin(['Student', 'Unemployed'])) &
                              (df['declared_income'] > 750000))).astype(int)
    df['rule_velocity'] = (df['num_applications'] >= 4).astype(int)
    df['rule_address'] = (df['address_change_count'] >= 4).astype(int)
    df['rule_score'] = df[['rule_age_exp', 'rule_income_job', 'rule_velocity', 'rule_address']].sum(axis=1)

    # Share features
    df['device_share'] = df.groupby('device_id')['applicant_id'].transform('count')
    df['ip_share'] = df.groupby('ip_address')['applicant_id'].transform('count')
    df['phone_share'] = df.groupby('phone_number')['applicant_id'].transform('count')

    df['device_share_log'] = np.log1p(df['device_share'])
    df['ip_share_log'] = np.log1p(df['ip_share'])
    df['phone_share_log'] = np.log1p(df['phone_share'])

    df['age_exp_gap'] = df['age'] - df['work_experience']
    df['income_per_exp'] = df['declared_income'] / (df['work_experience'] + 1)

    return df

applicants = apply_rules_and_features(applicants)

# --------------------------
# Identity Coherence Score
# --------------------------
def calculate_ics(df):
    df = df.copy()

    # Demographic Coherence
    demo_penalty = (
        df['rule_age_exp'] * 0.40 +
        df['rule_income_job'] * 0.35 +
        (df['age_exp_gap'] < 10).astype(int) * 0.15 +
        (df['income_per_exp'] > df['income_per_exp'].quantile(0.95)).astype(int) * 0.10
    )
    demo_penalty = np.clip(demo_penalty, 0, 1)
    df['demographic_coherence'] = 1 - demo_penalty

    # Relational Coherence
    rel_penalty = (
        0.50 * df['device_share_log'] +
        0.30 * df['ip_share_log'] +
        0.20 * df['phone_share_log']
    )
    scaler = MinMaxScaler()
    rel_penalty_norm = scaler.fit_transform(rel_penalty.values.reshape(-1, 1)).flatten()
    df['relational_coherence'] = 1 - rel_penalty_norm

    # Behavioral Coherence
    behav_penalty = (
        0.60 * (df['num_applications'] >= 4).astype(int) +
        0.40 * (df['address_change_count'] >= 3).astype(int)
    )
    behav_penalty = np.clip(behav_penalty, 0, 1)
    df['behavioral_coherence'] = 1 - behav_penalty

    # Final ICS
    df['identity_coherence_score'] = (
        0.40 * df['demographic_coherence'] +
        0.35 * df['relational_coherence'] +
        0.25 * df['behavioral_coherence']
    )

    df['identity_risk_from_ics'] = 1 - df['identity_coherence_score']
    return df

applicants = calculate_ics(applicants)

print("\nICS calculated on applicants")
print(applicants[['applicant_id', 'identity_coherence_score', 'identity_risk_from_ics', 'label']].head(8))
print("\nICS Stats:")
print(applicants['identity_coherence_score'].describe())

Applicants: (4000, 12)
Accounts  : (4906, 3)
Transactions: (59055, 4)

ICS calculated on applicants
  applicant_id  identity_coherence_score  identity_risk_from_ics  label
0    APP100000                  1.000000                0.000000      0
1    APP100001                  1.000000                0.000000      1
2    APP100002                  1.000000                0.000000      0
3    APP100003                  0.193487                0.806513      1
4    APP100004                  0.815066                0.184934      1
5    APP100005                  1.000000                0.000000      0
6    APP100006                  1.000000                0.000000      0
7    APP100007                  0.960000                0.040000      0

ICS Stats:
count    4000.000000
mean        0.940873
std         0.185427
min         0.064105
25%         1.000000
50%         1.000000
75%         1.000000
max         1.000000
Name: identity_coherence_score, dtype: float64


In [ ]:
# STEP 2: Diagnose why ICS equals 1.0

components = [
    'demographic_coherence',
    'relational_coherence',
    'behavioral_coherence',
    'identity_coherence_score'
]

print("Mean component scores by label:")
print(applicants.groupby('label')[components].mean().round(4))

print("\nPercentage of applicants with perfect component scores:")
for col in components:
    pct = (applicants[col] >= 0.999999).mean() * 100
    print(f"{col}: {pct:.2f}%")

print("\nApplicants with ICS = 1.0, by label:")
perfect = applicants[applicants['identity_coherence_score'] >= 0.999999]
print(perfect['label'].value_counts().sort_index())
print(
    perfect['label'].value_counts(normalize=True)
    .sort_index()
    .mul(100)
    .round(2)
)

print("\nDemographic rule overlap:")
print(pd.crosstab(
    applicants['rule_age_exp'],
    (applicants['age_exp_gap'] < 10).astype(int),
    rownames=['rule_age_exp'],
    colnames=['age_exp_gap < 10']
))

Mean component scores by label:
       demographic_coherence  relational_coherence  behavioral_coherence  \
label                                                                      
0                     0.9945                1.0000                 1.000   
1                     0.6868                0.6862                 0.598   

       identity_coherence_score  
label                            
0                        0.9978  
1                        0.6644  

Percentage of applicants with perfect component scores:
demographic_coherence: 87.30%
relational_coherence: 89.90%
behavioral_coherence: 90.95%
identity_coherence_score: 83.67%

Applicants with ICS = 1.0, by label:
label
0    3135
1     212
Name: count, dtype: int64
label
0    93.67
1     6.33
Name: proportion, dtype: float64

Demographic rule overlap:
age_exp_gap < 10     0   1
rule_age_exp              
0                 3723   0
1                  203  74


In [ ]:
# ===============================================================
# STEP 2: Map ICS from Applicants → Accounts
# ===============================================================

# Merge ICS into accounts
accounts = accounts.merge(
    applicants[['applicant_id', 'identity_coherence_score', 'identity_risk_from_ics']],
    on='applicant_id',
    how='left'
)

print("Accounts after ICS mapping:")
print(accounts.head(8))
print("\nNull check (should be 0):", accounts['identity_coherence_score'].isnull().sum())

# Save updated accounts
accounts.to_csv('linked_accounts_with_ics.csv', index=False)
print("\nSaved: linked_accounts_with_ics.csv")

Accounts after ICS mapping:
  account_id applicant_id  label  identity_coherence_score  \
0  ACC200000    APP100000      0                  1.000000   
1  ACC200001    APP100001      1                  1.000000   
2  ACC200002    APP100002      0                  1.000000   
3  ACC200003    APP100003      1                  0.193487   
4  ACC200004    APP100003      1                  0.193487   
5  ACC200005    APP100004      1                  0.815066   
6  ACC200006    APP100004      1                  0.815066   
7  ACC200007    APP100005      0                  1.000000   

   identity_risk_from_ics  
0                0.000000  
1                0.000000  
2                0.000000  
3                0.806513  
4                0.806513  
5                0.184934  
6                0.184934  
7                0.000000  

Null check (should be 0): 0

Saved: linked_accounts_with_ics.csv


In [ ]:
%pip install torch-geometric

In [ ]:
import pandas as pd
from sklearn.metrics import roc_auc_score

# Check how ICS differs by label
print("ICS statistics by account label:")
print(
    accounts.groupby("label")["identity_coherence_score"]
    .agg(["count", "mean", "std", "min", "max"])
    .round(4)
)

# Higher fraud risk should correspond to lower coherence.
# Reverse ICS so that a higher score means greater fraud risk.
labels = accounts["label"].to_numpy()
fraud_risk_score = 1 - accounts["identity_coherence_score"].to_numpy()

print("\nICS-only fraud ROC-AUC:")
print(round(roc_auc_score(labels, fraud_risk_score), 4))

print("\nICS value distribution:")
print(
    accounts["identity_coherence_score"]
    .round(2)
    .value_counts(normalize=True)
    .sort_index()
    .round(3)
)

ICS statistics by account label:
       count    mean     std     min  max
label                                    
0       3774  0.9978  0.0091  0.9600  1.0
1       1132  0.6549  0.3345  0.0641  1.0

ICS-only fraud ROC-AUC:
0.839

ICS value distribution:
identity_coherence_score
0.06    0.000
0.07    0.001
0.08    0.002
0.09    0.002
0.10    0.001
        ...  
0.89    0.001
0.90    0.008
0.94    0.006
0.96    0.045
1.00    0.798
Name: proportion, Length: 83, dtype: float64


In [ ]:
# ===============================================================
# STEP 3: Build Graph from Transactions
# ===============================================================

import torch
from torch_geometric.data import Data
from sklearn.preprocessing import LabelEncoder

# Create account to index mapping
account_list = accounts['account_id'].unique().tolist()
account_to_idx = {acc: idx for idx, acc in enumerate(account_list)}
idx_to_account = {idx: acc for acc, idx in account_to_idx.items()}

print("Total nodes (accounts):", len(account_to_idx))

# Build edge_index from transactions
senders = []
receivers = []

for _, row in transactions.iterrows():
    s = row['sender_account']
    r = row['receiver_account']
    if s in account_to_idx and r in account_to_idx:
        senders.append(account_to_idx[s])
        receivers.append(account_to_idx[r])

edge_index = torch.tensor([senders, receivers], dtype=torch.long)
print("Total edges:", edge_index.shape[1])

# Node features
# We will use: label is target, ICS is the special feature
# For basic features we create simple degree-based + ICS

# Calculate basic graph features
from collections import Counter
out_degree = Counter(senders)
in_degree = Counter(receivers)

node_features = []
node_labels = []
node_ics = []

for idx in range(len(account_to_idx)):
    acc = idx_to_account[idx]
    acc_row = accounts[accounts['account_id'] == acc].iloc[0]

    feat = [
        out_degree.get(idx, 0),
        in_degree.get(idx, 0),
        out_degree.get(idx, 0) + in_degree.get(idx, 0),   # total degree
        acc_row['identity_coherence_score'],               # ICS as feature
        acc_row['identity_risk_from_ics']
    ]
    node_features.append(feat)
    node_labels.append(acc_row['label'])
    node_ics.append(acc_row['identity_coherence_score'])

x = torch.tensor(node_features, dtype=torch.float)
y = torch.tensor(node_labels, dtype=torch.long)
ics = torch.tensor(node_ics, dtype=torch.float)

print("Node feature shape:", x.shape)
print("Labels shape:", y.shape)
print("ICS shape:", ics.shape)

# Create PyG Data object
data = Data(x=x, edge_index=edge_index, y=y)
data.ics = ics          # store ICS separately for coherence-aware model

print("\nGraph constructed successfully!")
print(data)

Total nodes (accounts): 4906
Total edges: 59055
Node feature shape: torch.Size([4906, 5])
Labels shape: torch.Size([4906])
ICS shape: torch.Size([4906])

Graph constructed successfully!
Data(x=[4906, 5], edge_index=[2, 59055], y=[4906], ics=[4906])


In [57]:

# STEP 8: Evaluate ICS Alone on the Existing Test Set

import numpy as np
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    precision_score,
    recall_score,
    f1_score
)

# Get true labels and ICS for the existing test nodes
y_test = data.y[test_idx].detach().cpu().numpy()
ics_test = data.ics[test_idx].detach().cpu().numpy()

# Lower ICS means higher fraud risk
risk_test = 1.0 - ics_test

# AUC metrics
print("ICS-ONLY TEST RESULTS")
print("-" * 35)
print("ROC-AUC:", round(roc_auc_score(y_test, risk_test), 4))
print("PR-AUC :", round(average_precision_score(y_test, risk_test), 4))

# Precision among the highest-risk test nodes
def precision_at_k(y_true, scores, k):
    k = min(k, len(y_true))
    top_indices = np.argsort(scores)[::-1][:k]
    return float(np.mean(y_true[top_indices]))

for k in [100, 500, 1000]:
    if k <= len(y_test):
        print(f"Precision@{k}: {precision_at_k(y_test, risk_test, k):.4f}")

# Top 1%, 2%, and 5% of test nodes
for pct in [1, 2, 5]:
    k = max(1, int(np.ceil(len(y_test) * pct / 100)))
    print(
        f"Precision@Top {pct}% (k={k}): "
        f"{precision_at_k(y_test, risk_test, k):.4f}"
    )


ICS-ONLY TEST RESULTS
-----------------------------------
ROC-AUC: 0.8468
PR-AUC : 0.7732
Precision@100: 1.0000
Precision@500: 0.2900
Precision@Top 1% (k=8): 1.0000
Precision@Top 2% (k=15): 1.0000
Precision@Top 5% (k=37): 1.0000


In [ ]:
# STEP 4: Research Data Audit

import pandas as pd

# 1. Required columns check
required_accounts = [
    "account_id",
    "applicant_id",
    "label",
    "identity_coherence_score",
    "identity_risk_from_ics"
]

required_transactions = ["sender_account", "receiver_account"]

print("Missing account columns:",
      set(required_accounts) - set(accounts.columns))

print("Missing transaction columns:",
      set(required_transactions) - set(transactions.columns))

# 2. Dataset sizes
print("\n--- DATASET SUMMARY ---")
print("Unique applicants:", accounts["applicant_id"].nunique())
print("Unique accounts:", accounts["account_id"].nunique())
print("Transaction records:", len(transactions))

# 3. Missing values
print("\n--- MISSING VALUES ---")
print(accounts[required_accounts].isnull().sum())

# 4. Duplicate account IDs
print("\nDuplicate account IDs:",
      accounts["account_id"].duplicated().sum())

# 5. Check whether accounts belonging to one applicant
# have consistent fraud labels
labels_per_applicant = (
    accounts.groupby("applicant_id")["label"].nunique()
)

print("\nApplicants with inconsistent account labels:",
      (labels_per_applicant > 1).sum())

# 6. Check whether ICS is consistent across an applicant's accounts
ics_per_applicant = (
    accounts.groupby("applicant_id")["identity_coherence_score"].nunique()
)

print("Applicants with inconsistent ICS:",
      (ics_per_applicant > 1).sum())

# 7. Check ICS values
print("\nICS minimum:", accounts["identity_coherence_score"].min())
print("ICS maximum:", accounts["identity_coherence_score"].max())

# 8. Check whether all transaction endpoints exist
known_accounts = set(accounts["account_id"])

unknown_senders = (~transactions["sender_account"].isin(known_accounts)).sum()
unknown_receivers = (~transactions["receiver_account"].isin(known_accounts)).sum()

print("\nTransactions with unknown senders:", unknown_senders)
print("Transactions with unknown receivers:", unknown_receivers)

print("\nAudit completed.")

Missing account columns: set()
Missing transaction columns: set()

--- DATASET SUMMARY ---
Unique applicants: 4000
Unique accounts: 4906
Transaction records: 59055

--- MISSING VALUES ---
account_id                  0
applicant_id                0
label                       0
identity_coherence_score    0
identity_risk_from_ics      0
dtype: int64

Duplicate account IDs: 0

Applicants with inconsistent account labels: 0
Applicants with inconsistent ICS: 0

ICS minimum: 0.0641049178665328
ICS maximum: 1.0

Transactions with unknown senders: 0
Transactions with unknown receivers: 0

Audit completed.


In [ ]:
# STEP 5: Applicant-Level Train / Validation / Test Split

import numpy as np
from sklearn.model_selection import train_test_split

SEED = 42

# One row per applicant
applicant_df = (
    accounts[["applicant_id", "label"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

# Confirm each applicant has exactly one label
assert applicant_df["applicant_id"].is_unique
assert applicant_df["label"].nunique() == 2

# 70% train, 15% validation, 15% test
train_df, temp_df = train_test_split(
    applicant_df,
    test_size=0.30,
    random_state=SEED,
    stratify=applicant_df["label"]
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=SEED,
    stratify=temp_df["label"]
)

# Map applicant IDs back to account-node indices
train_applicants = set(train_df["applicant_id"])
val_applicants = set(val_df["applicant_id"])
test_applicants = set(test_df["applicant_id"])

train_idx = [
    account_to_idx[acc]
    for acc in account_list
    if accounts.loc[
        accounts["account_id"] == acc, "applicant_id"
    ].iloc[0] in train_applicants
]

val_idx = [
    account_to_idx[acc]
    for acc in account_list
    if accounts.loc[
        accounts["account_id"] == acc, "applicant_id"
    ].iloc[0] in val_applicants
]

test_idx = [
    account_to_idx[acc]
    for acc in account_list
    if accounts.loc[
        accounts["account_id"] == acc, "applicant_id"
    ].iloc[0] in test_applicants
]

# Verify no applicant overlap
assert train_applicants.isdisjoint(val_applicants)
assert train_applicants.isdisjoint(test_applicants)
assert val_applicants.isdisjoint(test_applicants)

# Verify every account node is assigned exactly once
assert len(train_idx) + len(val_idx) + len(test_idx) == len(account_list)

print("Applicants:", len(applicant_df))
print("Train applicants:", len(train_applicants))
print("Validation applicants:", len(val_applicants))
print("Test applicants:", len(test_applicants))

print("\nAccount nodes:")
print("Train:", len(train_idx))
print("Validation:", len(val_idx))
print("Test:", len(test_idx))

print("\nFraud rate by applicant:")
print("Train:", round(train_df["label"].mean(), 4))
print("Validation:", round(val_df["label"].mean(), 4))
print("Test:", round(test_df["label"].mean(), 4))

print("\nApplicant-level split verified successfully.")

Applicants: 4000
Train applicants: 2800
Validation applicants: 600
Test applicants: 600

Account nodes:
Train: 3451
Validation: 728
Test: 727

Fraud rate by applicant:
Train: 0.1707
Validation: 0.17
Test: 0.1717

Applicant-level split verified successfully.


In [ ]:

# STEP 6A: Prepare Baseline GAT Features

import torch
from torch_geometric.data import Data

# Baseline uses only the first 3 graph features
x_baseline = data.x[:, :3].clone()

# Create baseline graph data
baseline_data = Data(
    x=x_baseline,
    edge_index=data.edge_index.clone(),
    y=data.y.clone()
)

# Convert split indices to PyTorch tensors
train_idx = torch.tensor(train_idx, dtype=torch.long)
val_idx = torch.tensor(val_idx, dtype=torch.long)
test_idx = torch.tensor(test_idx, dtype=torch.long)

print("Baseline feature shape:", baseline_data.x.shape)
print("Graph edge shape:", baseline_data.edge_index.shape)
print("Labels shape:", baseline_data.y.shape)

print("\nTrain nodes:", len(train_idx))
print("Validation nodes:", len(val_idx))
print("Test nodes:", len(test_idx))

assert (
    len(train_idx) + len(val_idx) + len(test_idx)
    == baseline_data.num_nodes
)

print("\nBaseline GAT data prepared successfully!")


Baseline feature shape: torch.Size([4906, 3])
Graph edge shape: torch.Size([2, 59055])
Labels shape: torch.Size([4906])

Train nodes: 3451
Validation nodes: 728
Test nodes: 727

Baseline GAT data prepared successfully!


In [ ]:

# STEP 6B: Baseline GAT Model

import torch
import torch.nn.functional as F
from torch_geometric.nn import GATConv
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix
)

# Reproducibility
torch.manual_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

class BaselineGAT(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=16, heads=2):
        super().__init__()

        self.gat1 = GATConv(
            in_channels, hidden_channels,
            heads=heads, dropout=0.3
        )
        self.gat2 = GATConv(
            hidden_channels * heads, 2,
            heads=1, concat=False, dropout=0.3
        )

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.gat1(x, edge_index)
        x = F.elu(x)
        x = F.dropout(x, p=0.3, training=self.training)
        x = self.gat2(x, edge_index)
        return x

# Prepare model and data
baseline_data = baseline_data.to(device)
train_idx = train_idx.to(device)
val_idx = val_idx.to(device)
test_idx = test_idx.to(device)

model = BaselineGAT(
    in_channels=baseline_data.x.shape[1]
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(), lr=0.005, weight_decay=5e-4
)

# Train using training labels only
best_val_f1 = -1
best_state = None
patience = 30
wait = 0

for epoch in range(1, 301):
    model.train()
    optimizer.zero_grad()

    logits = model(baseline_data.x, baseline_data.edge_index)
    loss = F.cross_entropy(
        logits[train_idx], baseline_data.y[train_idx]
    )

    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(
            baseline_data.x, baseline_data.edge_index
        )
        val_probs = torch.softmax(val_logits[val_idx], dim=1)[:, 1]
        val_preds = (val_probs >= 0.5).long()

        val_f1 = f1_score(
            baseline_data.y[val_idx].cpu().numpy(),
            val_preds.cpu().numpy(),
            zero_division=0
        )

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model.state_dict().items()
        }
        wait = 0
    else:
        wait += 1

    if epoch == 1 or epoch % 20 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss.item():.4f} | "
            f"Validation F1: {val_f1:.4f}"
        )

    if wait >= patience:
        print("Early stopping at epoch:", epoch)
        break

# Restore best validation checkpoint
model.load_state_dict(best_state)
model.eval()

# Evaluate on test set only after training/model selection
with torch.no_grad():
    test_logits = model(
        baseline_data.x, baseline_data.edge_index
    )
    test_probs = torch.softmax(
        test_logits[test_idx], dim=1
    )[:, 1].cpu().numpy()

test_labels = baseline_data.y[test_idx].cpu().numpy()
test_preds = (test_probs >= 0.5).astype(int)

print("\n--- BASELINE GAT TEST RESULTS ---")
print("Accuracy:", round(accuracy_score(test_labels, test_preds), 4))
print("Precision:", round(precision_score(test_labels, test_preds, zero_division=0), 4))
print("Recall:", round(recall_score(test_labels, test_preds, zero_division=0), 4))
print("F1-score:", round(f1_score(test_labels, test_preds, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(test_labels, test_probs), 4))
print("PR-AUC:", round(average_precision_score(test_labels, test_probs), 4))
print("Confusion Matrix:\n", confusion_matrix(test_labels, test_preds))


Using device: cpu
Epoch 001 | Loss: 2.6382 | Validation F1: 0.0000
Epoch 020 | Loss: 0.7864 | Validation F1: 0.0000
Early stopping at epoch: 31

--- BASELINE GAT TEST RESULTS ---
Accuracy: 0.7772
Precision: 0.0
Recall: 0.0
F1-score: 0.0
ROC-AUC: 0.2864
PR-AUC: 0.1501
Confusion Matrix:
 [[565   0]
 [162   0]]


In [ ]:

# STEP 7: Diagnose Baseline GAT

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import roc_auc_score

model.eval()

with torch.no_grad():
    logits = model(baseline_data.x, baseline_data.edge_index)
    all_probs = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()

all_labels = baseline_data.y.cpu().numpy()
all_features = baseline_data.x.cpu().numpy()

train_ids = train_idx.cpu().numpy()
val_ids = val_idx.cpu().numpy()
test_ids = test_idx.cpu().numpy()

print("1. LABEL DISTRIBUTION")
print(pd.Series(all_labels).value_counts().sort_index())
print("Interpretation: confirm which number represents fraud.")

print("\n2. FEATURE RANGES")
print(pd.DataFrame(
    all_features,
    columns=["out_degree", "in_degree", "total_degree"]
).describe().round(2))

print("\n3. MEAN FEATURES BY LABEL")
feature_df = pd.DataFrame(
    all_features,
    columns=["out_degree", "in_degree", "total_degree"]
)
feature_df["label"] = all_labels
print(feature_df.groupby("label").mean().round(2))

print("\n4. PREDICTED FRAUD PROBABILITIES")
for name, ids in [
    ("Train", train_ids),
    ("Validation", val_ids),
    ("Test", test_ids)
]:
    probs = all_probs[ids]
    labels = all_labels[ids]

    print(f"\n{name}:")
    print("Actual fraud rate:", round(labels.mean(), 4))
    print("Mean predicted fraud probability:", round(probs.mean(), 4))
    print("Min probability:", round(probs.min(), 4))
    print("Max probability:", round(probs.max(), 4))

    for label in [0, 1]:
        mask = labels == label
        if mask.sum():
            print(
                f"Mean probability for actual label {label}:",
                round(probs[mask].mean(), 4)
            )

    if len(np.unique(labels)) == 2:
        print(
            "ROC-AUC:",
            round(roc_auc_score(labels, probs), 4)
        )

print("\nDiagnostic completed.")


1. LABEL DISTRIBUTION
0    3774
1    1132
Name: count, dtype: int64
Interpretation: confirm which number represents fraud.

2. FEATURE RANGES
       out_degree  in_degree  total_degree
count     4906.00    4906.00       4906.00
mean        12.04      12.04         24.07
std          2.58       4.46          5.18
min          8.00       1.00         10.00
25%         10.00       9.00         20.00
50%         12.00      11.00         24.00
75%         14.00      15.00         27.00
max         16.00      32.00         44.00

3. MEAN FEATURES BY LABEL
       out_degree  in_degree  total_degree
label                                     
0           12.00  10.490000     22.490000
1           12.17  17.190001     29.370001

4. PREDICTED FRAUD PROBABILITIES

Train:
Actual fraud rate: 0.2333
Mean predicted fraud probability: 0.0008
Min probability: 0.0005
Max probability: 0.0031
Mean probability for actual label 0: 0.0008
Mean probability for actual label 1: 0.0007
ROC-AUC: 0.3292

Validation

In [ ]:

# STEP 8: Corrected Baseline GAT Training

import copy
import numpy as np
import torch
import torch.nn.functional as F

from torch_geometric.nn import GATConv
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Move graph and splits to device
graph = baseline_data.to(device)
tr = train_idx.to(device)
va = val_idx.to(device)
te = test_idx.to(device)

# Standardize using TRAINING nodes only
train_x = graph.x[tr]
feature_mean = train_x.mean(dim=0)
feature_std = train_x.std(dim=0).clamp(min=1e-6)

x_scaled = (graph.x - feature_mean) / feature_std

# Class weights calculated from training labels only
train_labels = graph.y[tr]
counts = torch.bincount(train_labels, minlength=2).float()
class_weights = counts.sum() / (2 * counts)

print("Training label counts:", counts.cpu().tolist())
print("Class weights:", class_weights.cpu().tolist())

class BaselineGATv2(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=16, heads=2):
        super().__init__()
        self.gat1 = GATConv(
            in_channels, hidden_channels,
            heads=heads, dropout=0.3
        )
        self.gat2 = GATConv(
            hidden_channels * heads, 2,
            heads=1, concat=False, dropout=0.3
        )

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.elu(self.gat1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        return self.gat2(x, edge_index)

model2 = BaselineGATv2(x_scaled.shape[1]).to(device)

optimizer = torch.optim.Adam(
    model2.parameters(), lr=0.005, weight_decay=5e-4
)

weights = class_weights.to(device)
best_val_ap = -1
best_state = None
patience = 40
wait = 0

for epoch in range(1, 301):
    model2.train()
    optimizer.zero_grad()

    logits = model2(x_scaled, graph.edge_index)
    loss = F.cross_entropy(
        logits[tr], graph.y[tr], weight=weights
    )
    loss.backward()
    optimizer.step()

    model2.eval()
    with torch.no_grad():
        val_logits = model2(x_scaled, graph.edge_index)
        val_probs = torch.softmax(val_logits[va], dim=1)[:, 1]
        val_labels = graph.y[va].cpu().numpy()
        val_probs_np = val_probs.cpu().numpy()

        val_ap = average_precision_score(val_labels, val_probs_np)

    if val_ap > best_val_ap + 1e-5:
        best_val_ap = val_ap
        best_state = copy.deepcopy(model2.state_dict())
        wait = 0
    else:
        wait += 1

    if epoch == 1 or epoch % 20 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss.item():.4f} | "
            f"Validation PR-AUC: {val_ap:.4f}"
        )

    if wait >= patience:
        print("Early stopping at epoch:", epoch)
        break

# Restore checkpoint selected on validation PR-AUC
model2.load_state_dict(best_state)
model2.eval()

with torch.no_grad():
    test_logits = model2(x_scaled, graph.edge_index)
    test_probs = torch.softmax(test_logits[te], dim=1)[:, 1].cpu().numpy()

test_labels = graph.y[te].cpu().numpy()
test_preds = (test_probs >= 0.5).astype(int)

print("\n--- CORRECTED BASELINE TEST RESULTS ---")
print("Accuracy:", round(accuracy_score(test_labels, test_preds), 4))
print("Precision:", round(precision_score(test_labels, test_preds, zero_division=0), 4))
print("Recall:", round(recall_score(test_labels, test_preds, zero_division=0), 4))
print("F1:", round(f1_score(test_labels, test_preds, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(test_labels, test_probs), 4))
print("PR-AUC:", round(average_precision_score(test_labels, test_probs), 4))
print("Confusion matrix:\n", confusion_matrix(test_labels, test_preds))

print("\nMean predicted fraud probability:",
      round(float(test_probs.mean()), 4))
print("Min / max predicted probability:",
      round(float(test_probs.min()), 4),
      round(float(test_probs.max()), 4))


Training label counts: [2646.0, 805.0]
Class weights: [0.6521164178848267, 2.1434781551361084]
Epoch 001 | Loss: 0.7449 | Validation PR-AUC: 0.1346
Epoch 020 | Loss: 0.5582 | Validation PR-AUC: 0.8205
Epoch 040 | Loss: 0.4193 | Validation PR-AUC: 0.8584
Epoch 060 | Loss: 0.3875 | Validation PR-AUC: 0.8793
Epoch 080 | Loss: 0.3643 | Validation PR-AUC: 0.8936
Epoch 100 | Loss: 0.3750 | Validation PR-AUC: 0.9011
Epoch 120 | Loss: 0.3664 | Validation PR-AUC: 0.9034
Epoch 140 | Loss: 0.3657 | Validation PR-AUC: 0.9068
Epoch 160 | Loss: 0.3767 | Validation PR-AUC: 0.9048
Epoch 180 | Loss: 0.3550 | Validation PR-AUC: 0.9084
Epoch 200 | Loss: 0.3469 | Validation PR-AUC: 0.9099
Epoch 220 | Loss: 0.3663 | Validation PR-AUC: 0.9105
Epoch 240 | Loss: 0.3649 | Validation PR-AUC: 0.9123
Epoch 260 | Loss: 0.3443 | Validation PR-AUC: 0.9113
Epoch 280 | Loss: 0.3487 | Validation PR-AUC: 0.9104
Epoch 300 | Loss: 0.3422 | Validation PR-AUC: 0.9127

--- CORRECTED BASELINE TEST RESULTS ---
Accuracy: 0.934


In [ ]:

# STEP 9: GAT + ICS Features
# Same split and training procedure as the corrected baseline

import copy
import numpy as np
import torch
import torch.nn.functional as F

from torch_geometric.nn import GATConv
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Original graph has 5 features:
# out_degree, in_degree, total_degree, ICS, ICS-derived risk
ics_graph = data.to(device)
tr = train_idx.to(device)
va = val_idx.to(device)
te = test_idx.to(device)

# Standardize using TRAINING nodes only
train_x = ics_graph.x[tr]
feature_mean_ics = train_x.mean(dim=0)
feature_std_ics = train_x.std(dim=0).clamp(min=1e-6)
x_scaled_ics = (ics_graph.x - feature_mean_ics) / feature_std_ics

# Use class weights calculated from training labels only
train_labels = ics_graph.y[tr]
counts = torch.bincount(train_labels, minlength=2).float()
class_weights_ics = counts.sum() / (2 * counts)
weights = class_weights_ics.to(device)

class GATWithICS(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels=16, heads=2):
        super().__init__()
        self.gat1 = GATConv(
            in_channels, hidden_channels,
            heads=heads, dropout=0.3
        )
        self.gat2 = GATConv(
            hidden_channels * heads, 2,
            heads=1, concat=False, dropout=0.3
        )

    def forward(self, x, edge_index):
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.elu(self.gat1(x, edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        return self.gat2(x, edge_index)

model_ics = GATWithICS(
    in_channels=x_scaled_ics.shape[1]
).to(device)

optimizer = torch.optim.Adam(
    model_ics.parameters(), lr=0.005, weight_decay=5e-4
)

best_val_ap_ics = -1
best_state_ics = None
patience = 40
wait = 0

for epoch in range(1, 301):
    model_ics.train()
    optimizer.zero_grad()

    logits = model_ics(x_scaled_ics, ics_graph.edge_index)
    loss = F.cross_entropy(
        logits[tr], ics_graph.y[tr], weight=weights
    )
    loss.backward()
    optimizer.step()

    model_ics.eval()
    with torch.no_grad():
        val_logits = model_ics(
            x_scaled_ics, ics_graph.edge_index
        )
        val_probs = torch.softmax(val_logits[va], dim=1)[:, 1]
        val_labels = ics_graph.y[va].cpu().numpy()
        val_probs_np = val_probs.cpu().numpy()

        val_ap = average_precision_score(
            val_labels, val_probs_np
        )

    if val_ap > best_val_ap_ics + 1e-5:
        best_val_ap_ics = val_ap
        best_state_ics = copy.deepcopy(model_ics.state_dict())
        wait = 0
    else:
        wait += 1

    if epoch == 1 or epoch % 20 == 0:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss.item():.4f} | "
            f"Validation PR-AUC: {val_ap:.4f}"
        )

    if wait >= patience:
        print("Early stopping at epoch:", epoch)
        break

# Restore best validation checkpoint
model_ics.load_state_dict(best_state_ics)
model_ics.eval()

# Evaluate test set only after model selection
with torch.no_grad():
    test_logits = model_ics(
        x_scaled_ics, ics_graph.edge_index
    )
    test_probs_ics = torch.softmax(
        test_logits[te], dim=1
    )[:, 1].cpu().numpy()

test_labels_ics = ics_graph.y[te].cpu().numpy()
test_preds_ics = (test_probs_ics >= 0.5).astype(int)

print("\n--- GAT + ICS TEST RESULTS ---")
print("Accuracy:", round(accuracy_score(test_labels_ics, test_preds_ics), 4))
print("Precision:", round(precision_score(test_labels_ics, test_preds_ics, zero_division=0), 4))
print("Recall:", round(recall_score(test_labels_ics, test_preds_ics, zero_division=0), 4))
print("F1:", round(f1_score(test_labels_ics, test_preds_ics, zero_division=0), 4))
print("ROC-AUC:", round(roc_auc_score(test_labels_ics, test_probs_ics), 4))
print("PR-AUC:", round(average_precision_score(test_labels_ics, test_probs_ics), 4))
print("Confusion matrix:\n", confusion_matrix(test_labels_ics, test_preds_ics))


Epoch 001 | Loss: 1.0595 | Validation PR-AUC: 0.1364
Epoch 020 | Loss: 0.6184 | Validation PR-AUC: 0.6632
Epoch 040 | Loss: 0.5210 | Validation PR-AUC: 0.6998
Epoch 060 | Loss: 0.4702 | Validation PR-AUC: 0.7010
Epoch 080 | Loss: 0.4420 | Validation PR-AUC: 0.7570
Epoch 100 | Loss: 0.3845 | Validation PR-AUC: 0.7994
Epoch 120 | Loss: 0.3467 | Validation PR-AUC: 0.8520
Epoch 140 | Loss: 0.3428 | Validation PR-AUC: 0.8670
Epoch 160 | Loss: 0.3269 | Validation PR-AUC: 0.8747
Epoch 180 | Loss: 0.3225 | Validation PR-AUC: 0.8794
Epoch 200 | Loss: 0.3276 | Validation PR-AUC: 0.8819
Epoch 220 | Loss: 0.3207 | Validation PR-AUC: 0.8829
Epoch 240 | Loss: 0.3191 | Validation PR-AUC: 0.8862
Epoch 260 | Loss: 0.3215 | Validation PR-AUC: 0.8870
Epoch 280 | Loss: 0.3141 | Validation PR-AUC: 0.8873
Epoch 300 | Loss: 0.3300 | Validation PR-AUC: 0.8901

--- GAT + ICS TEST RESULTS ---
Accuracy: 0.9532
Precision: 0.8478
Recall: 0.963
F1: 0.9017
ROC-AUC: 0.9853
PR-AUC: 0.9208
Confusion matrix:
 [[537  28]

In [ ]:
import sys
import torch

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

try:
    import torch_geometric
    print("PyTorch Geometric:", torch_geometric.__version__)
except ImportError:
    print("PyTorch Geometric: NOT INSTALLED")

Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cpu
GPU available: False
PyTorch Geometric: 2.8.1


In [ ]:
%pip install -q torch-geometric

In [ ]:
import torch
import torch_geometric

print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__)
print("GPU available:", torch.cuda.is_available())

PyTorch: 2.11.0+cpu
PyG: 2.8.1
GPU available: False


In [ ]:
import pandas as pd
import numpy as np
import networkx as nx
import torch
import torch_geometric
from torch_geometric.nn import GATConv

print("All core graph libraries imported successfully!")
print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__)

All core graph libraries imported successfully!
PyTorch: 2.11.0+cpu
PyG: 2.8.1


In [ ]:
print("data exists:", "data" in globals())
print("train_idx exists:", "train_idx" in globals())
print("val_idx exists:", "val_idx" in globals())
print("test_idx exists:", "test_idx" in globals())

if "data" in globals():
    print("Node features:", data.x.shape)
    print("Graph edges:", data.edge_index.shape)
    print("Labels:", data.y.shape)
    print("ICS exists:", hasattr(data, "ics"))

data exists: True
train_idx exists: True
val_idx exists: True
test_idx exists: True
Node features: torch.Size([4906, 5])
Graph edges: torch.Size([2, 59055])
Labels: torch.Size([4906])
ICS exists: True


In [ ]:
import torch

y = data.y.detach().cpu().view(-1).long()

for name, idx in [
    ("Train", train_idx),
    ("Validation", val_idx),
    ("Test", test_idx),
]:
    idx = torch.as_tensor(idx).detach().cpu().long().view(-1)
    labels = y[idx]

    print(f"\n{name}:")
    print("Nodes:", len(idx))
    print("Fraud:", int((labels == 1).sum()))
    print("Legitimate:", int((labels == 0).sum()))
    print("Fraud rate:", round(float((labels == 1).float().mean()) * 100, 2), "%")

print("\nTotal labels:", len(y))
print("Unique labels:", torch.unique(y).tolist())


Train:
Nodes: 3451
Fraud: 805
Legitimate: 2646
Fraud rate: 23.33 %

Validation:
Nodes: 728
Fraud: 165
Legitimate: 563
Fraud rate: 22.66 %

Test:
Nodes: 727
Fraud: 162
Legitimate: 565
Fraud rate: 22.28 %

Total labels: 4906
Unique labels: [0, 1]


In [ ]:
print("Existing model variables:")
for name in ["model", "model_ics", "data", "train_idx", "val_idx", "test_idx"]:
    print(name, "exists:", name in globals())

print("\nExisting metric variables:")
for name in ["test_auc", "test_f1", "results", "history"]:
    print(name, "exists:", name in globals())

Existing model variables:
model exists: True
model_ics exists: True
data exists: True
train_idx exists: True
val_idx exists: True
test_idx exists: True

Existing metric variables:
test_auc exists: False
test_f1 exists: False
results exists: False
history exists: False


In [ ]:
print("Baseline model:")
print(model)

print("\nGAT + ICS model:")
print(model_ics)

print("\nGraph feature shape:", tuple(data.x.shape))
print("Graph edge shape:", tuple(data.edge_index.shape))
print("ICS shape:", tuple(data.ics.shape))

Baseline model:
BaselineGAT(
  (gat1): GATConv(3, 16, heads=2)
  (gat2): GATConv(32, 2, heads=1)
)

GAT + ICS model:
GATWithICS(
  (gat1): GATConv(5, 16, heads=2)
  (gat2): GATConv(32, 2, heads=1)
)

Graph feature shape: (4906, 5)
Graph edge shape: (2, 59055)
ICS shape: (4906,)


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.nn import MessagePassing
from torch_geometric.utils import softmax


class CoherenceAttentionLayer(MessagePassing):
    def __init__(self, in_channels, out_channels, heads=2,
                 concat=True, coherence_lambda=1.0, dropout=0.2):
        super().__init__(aggr="add", node_dim=0)

        self.heads = heads
        self.out_channels = out_channels
        self.concat = concat
        self.coherence_lambda = coherence_lambda
        self.dropout = dropout

        self.lin = nn.Linear(in_channels, heads * out_channels, bias=False)
        self.att_src = nn.Parameter(torch.empty(1, heads, out_channels))
        self.att_dst = nn.Parameter(torch.empty(1, heads, out_channels))
        self.bias = nn.Parameter(
            torch.zeros(heads * out_channels if concat else out_channels)
        )

        self.reset_parameters()

    def reset_parameters(self):
        nn.init.xavier_uniform_(self.lin.weight)
        nn.init.xavier_uniform_(self.att_src)
        nn.init.xavier_uniform_(self.att_dst)
        nn.init.zeros_(self.bias)

    def forward(self, x, edge_index, ics):
        n = x.size(0)

        h = self.lin(x).view(n, self.heads, self.out_channels)

        src, dst = edge_index
        score = (
            (h[src] * self.att_src).sum(dim=-1)
            + (h[dst] * self.att_dst).sum(dim=-1)
        )
        score = F.leaky_relu(score, negative_slope=0.2)

        # Lower ICS means greater identity incoherence.
        # The bias encourages attention between two low-ICS nodes.
        ics = ics.reshape(-1).to(x.device).float().clamp(0, 1)
        incoherence = 1.0 - ics
        coherence_bias = (
            self.coherence_lambda
            * incoherence[src]
            * incoherence[dst]
        ).unsqueeze(-1)

        alpha = softmax(score + coherence_bias, dst, num_nodes=n)
        alpha = F.dropout(alpha, p=self.dropout, training=self.training)

        out = self.propagate(edge_index, x=h, alpha=alpha, size=(n, n))

        if self.concat:
            out = out.reshape(n, self.heads * self.out_channels)
        else:
            out = out.mean(dim=1)

        return out + self.bias

    def message(self, x_j, alpha):
        return x_j * alpha.unsqueeze(-1)


class CoherenceAwareGAT(nn.Module):
    def __init__(self, in_channels=3, hidden_channels=16,
                 heads=2, coherence_lambda=1.0, dropout=0.2):
        super().__init__()

        self.gat1 = CoherenceAttentionLayer(
            in_channels, hidden_channels,
            heads=heads, concat=True,
            coherence_lambda=coherence_lambda,
            dropout=dropout
        )

        self.gat2 = CoherenceAttentionLayer(
            hidden_channels * heads, 2,
            heads=1, concat=False,
            coherence_lambda=coherence_lambda,
            dropout=dropout
        )

        self.dropout = dropout

    def forward(self, x, edge_index, ics):
        x = self.gat1(x, edge_index, ics)
        x = F.elu(x)
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = self.gat2(x, edge_index, ics)
        return x

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x_raw = data.x.detach().cpu().float()
edge_index = data.edge_index.detach().cpu().long()
ics_raw = data.ics.detach().cpu().float().reshape(-1)
y = data.y.detach().cpu().long().reshape(-1)

coherence_model = CoherenceAwareGAT(
    in_channels=3,
    hidden_channels=16,
    heads=2,
    coherence_lambda=1.0
).to(device)

coherence_model.eval()

with torch.no_grad():
    sample_logits = coherence_model(
        x_raw[:, :3].to(device),
        edge_index.to(device),
        ics_raw.to(device)
    )

print("Device:", device)
print("Output shape:", tuple(sample_logits.shape))
print("Expected shape:", (len(y), 2))
print("Model created successfully!")

Device: cpu
Output shape: (4906, 2)
Expected shape: (4906, 2)
Model created successfully!


In [ ]:
import inspect

print("BaselineGAT constructor:")
print(inspect.signature(BaselineGAT.__init__))

print("\nGATWithICS constructor:")
print(inspect.signature(GATWithICS.__init__))

print("\nBaseline forward method:")
print(inspect.signature(BaselineGAT.forward))

print("\nGAT + ICS forward method:")
print(inspect.signature(GATWithICS.forward))

BaselineGAT constructor:
(self, in_channels, hidden_channels=16, heads=2)

GATWithICS constructor:
(self, in_channels, hidden_channels=16, heads=2)

Baseline forward method:
(self, x, edge_index)

GAT + ICS forward method:
(self, x, edge_index)


In [ ]:
import copy
import os
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
)
from sklearn.preprocessing import StandardScaler

# -----------------------------
# 1. Reproducibility and data
# -----------------------------
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Training device:", device)

x_all = data.x.detach().cpu().float().numpy()
edge_index = data.edge_index.detach().cpu().long()
ics_all = data.ics.detach().cpu().float().reshape(-1)
y_all = data.y.detach().cpu().long().reshape(-1)

train_ids = torch.as_tensor(train_idx).detach().cpu().long().reshape(-1)
val_ids = torch.as_tensor(val_idx).detach().cpu().long().reshape(-1)
test_ids = torch.as_tensor(test_idx).detach().cpu().long().reshape(-1)

# Scale each feature using TRAIN nodes only.
# This does not remove possible information leakage from graph construction.
scaler = StandardScaler()
scaler.fit(x_all[train_ids.numpy()])

x_scaled = torch.tensor(
    scaler.transform(x_all), dtype=torch.float32
)

# Same graph and splits for all experiments.
edge_index = edge_index.to(device)
y_all = y_all.to(device)
train_ids = train_ids.to(device)
val_ids = val_ids.to(device)
test_ids = test_ids.to(device)
ics_all = ics_all.to(device)

# -----------------------------
# 2. Model configurations
# -----------------------------
def make_baseline():
    return BaselineGAT(
        in_channels=3, hidden_channels=16, heads=2
    )

def make_ics_model():
    return GATWithICS(
        in_channels=5, hidden_channels=16, heads=2
    )

def make_coherence_model():
    return CoherenceAwareGAT(
        in_channels=3,
        hidden_channels=16,
        heads=2,
        coherence_lambda=1.0,
        dropout=0.2,
    )

experiments = {
    "Baseline GAT": (make_baseline, 3, False),
    "GAT + ICS features": (make_ics_model, 5, False),
    "Coherence-Aware GAT": (make_coherence_model, 3, True),
}

# Balanced class weighting computed from training labels only.
train_y = y_all[train_ids]
n_positive = int((train_y == 1).sum())
n_negative = int((train_y == 0).sum())
pos_weight = torch.tensor(
    [n_negative / max(n_positive, 1)],
    dtype=torch.float32,
    device=device,
)

print("Training positives:", n_positive)
print("Training negatives:", n_negative)

# -----------------------------
# 3. Train one experiment
# -----------------------------
def train_experiment(factory, n_features, uses_ics):
    x = x_scaled[:, :n_features].to(device)
    model_new = factory().to(device)

    optimizer = torch.optim.Adam(
        model_new.parameters(), lr=0.005, weight_decay=5e-4
    )
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    best_pr_auc = -1.0
    best_state = None
    best_epoch = 0
    patience = 8
    epochs_without_improvement = 0
    max_epochs = 40

    for epoch in range(1, max_epochs + 1):
        model_new.train()
        optimizer.zero_grad()

        if uses_ics:
            logits = model_new(x, edge_index, ics_all)
        else:
            logits = model_new(x, edge_index)

        loss = criterion(logits[train_ids, 1], y_all[train_ids].float())
        loss.backward()
        optimizer.step()

        model_new.eval()
        with torch.no_grad():
            if uses_ics:
                val_logits = model_new(x, edge_index, ics_all)
            else:
                val_logits = model_new(x, edge_index)

            val_prob = torch.softmax(val_logits[val_ids], dim=1)[:, 1]
            val_prob_np = val_prob.cpu().numpy()
            val_y_np = y_all[val_ids].cpu().numpy()

            val_pr_auc = average_precision_score(val_y_np, val_prob_np)

        if val_pr_auc > best_pr_auc:
            best_pr_auc = val_pr_auc
            best_state = copy.deepcopy({
                k: v.detach().cpu().clone()
                for k, v in model_new.state_dict().items()
            })
            best_epoch = epoch
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Epoch {epoch:02d} | "
                f"loss={loss.item():.4f} | "
                f"val PR-AUC={val_pr_auc:.4f}"
            )

        if epochs_without_improvement >= patience:
            print("Early stopping.")
            break

    model_new.load_state_dict(best_state)
    model_new.eval()

    with torch.no_grad():
        if uses_ics:
            all_logits = model_new(x, edge_index, ics_all)
        else:
            all_logits = model_new(x, edge_index)

        val_prob = torch.softmax(all_logits[val_ids], dim=1)[:, 1]
        test_prob = torch.softmax(all_logits[test_ids], dim=1)[:, 1]

    val_prob = val_prob.cpu().numpy()
    test_prob = test_prob.cpu().numpy()
    val_y_np = y_all[val_ids].cpu().numpy()
    test_y_np = y_all[test_ids].cpu().numpy()

    # Select threshold using validation F1 only.
    precision, recall, thresholds = precision_recall_curve(
        val_y_np, val_prob
    )
    if len(thresholds):
        f1_values = (
            2 * precision[:-1] * recall[:-1]
            / (precision[:-1] + recall[:-1] + 1e-12)
        )
        best_threshold = float(thresholds[int(np.argmax(f1_values))])
    else:
        best_threshold = 0.5

    test_pred = (test_prob >= best_threshold).astype(int)

    metrics = {
        "Best epoch": best_epoch,
        "Best validation PR-AUC": best_pr_auc,
        "Threshold (validation)": best_threshold,
        "Test accuracy": accuracy_score(test_y_np, test_pred),
        "Test precision": precision_score(
            test_y_np, test_pred, zero_division=0
        ),
        "Test recall": recall_score(
            test_y_np, test_pred, zero_division=0
        ),
        "Test F1": f1_score(test_y_np, test_pred, zero_division=0),
        "Test ROC-AUC": roc_auc_score(test_y_np, test_prob),
        "Test PR-AUC": average_precision_score(test_y_np, test_prob),
    }

    return model_new, metrics

# -----------------------------
# 4. Run the three experiments
# -----------------------------
trained_models = {}
comparison_rows = []

for name, (factory, n_features, uses_ics) in experiments.items():
    print("\n" + "=" * 60)
    print(name)
    print("=" * 60)

    trained_model, metrics = train_experiment(
        factory, n_features, uses_ics
    )

    trained_models[name] = trained_model
    comparison_rows.append({"Model": name, **metrics})

comparison_df = pd.DataFrame(comparison_rows)

print("\nFINAL COMPARISON")
display(comparison_df.round(4))

Training device: cpu
Training positives: 805
Training negatives: 2646

Baseline GAT
Epoch 01 | loss=1.1031 | val PR-AUC=0.1351
Epoch 05 | loss=1.0657 | val PR-AUC=0.2632
Epoch 10 | loss=1.0121 | val PR-AUC=0.7160
Epoch 15 | loss=0.9820 | val PR-AUC=0.7624
Epoch 20 | loss=0.9306 | val PR-AUC=0.8014
Epoch 25 | loss=0.8889 | val PR-AUC=0.8167
Epoch 30 | loss=0.8136 | val PR-AUC=0.8221
Epoch 35 | loss=0.7966 | val PR-AUC=0.8233
Epoch 40 | loss=0.7440 | val PR-AUC=0.8318

GAT + ICS features
Epoch 01 | loss=1.0956 | val PR-AUC=0.5661
Epoch 05 | loss=1.0322 | val PR-AUC=0.6143
Epoch 10 | loss=0.9933 | val PR-AUC=0.6311
Epoch 15 | loss=0.9352 | val PR-AUC=0.6946
Epoch 20 | loss=0.8397 | val PR-AUC=0.7842
Epoch 25 | loss=0.7661 | val PR-AUC=0.8611
Epoch 30 | loss=0.6951 | val PR-AUC=0.8966
Epoch 35 | loss=0.6458 | val PR-AUC=0.9212
Epoch 40 | loss=0.5983 | val PR-AUC=0.9241

Coherence-Aware GAT
Epoch 01 | loss=1.0170 | val PR-AUC=0.6721
Epoch 05 | loss=0.9751 | val PR-AUC=0.5690
Early stopping.

,Model,Best epoch,Best validation PR-AUC,Threshold (validation),Test accuracy,Test precision,Test recall,Test F1,Test ROC-AUC,Test PR-AUC
0,Baseline GAT,39,0.8354,0.4124,0.9051,0.7514,0.8580,0.8012,0.9631,0.8424
1,GAT + ICS features,38,0.9296,0.5375,0.9436,0.9172,0.8210,0.8664,0.9815,0.9150
2,Coherence-Aware GAT,1,0.6721,0.5297,0.8501,0.6268,0.8086,0.7062,0.9164,0.6961


In [ ]:
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    average_precision_score, roc_auc_score,
    precision_recall_curve, accuracy_score,
    precision_score, recall_score, f1_score
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Reuse the previously prepared data and train-only-scaled features.
x_scaled = x_scaled.detach().cpu().float()
edge_index_cpu = data.edge_index.detach().cpu().long()
ics_cpu = data.ics.detach().cpu().float().reshape(-1)
y_cpu = data.y.detach().cpu().long().reshape(-1)

train_cpu = torch.as_tensor(train_idx).detach().cpu().long().reshape(-1)
val_cpu = torch.as_tensor(val_idx).detach().cpu().long().reshape(-1)
test_cpu = torch.as_tensor(test_idx).detach().cpu().long().reshape(-1)

edge_index = edge_index_cpu.to(device)
ics = ics_cpu.to(device)
y = y_cpu.to(device)
train_ids = train_cpu.to(device)
val_ids = val_cpu.to(device)
test_ids = test_cpu.to(device)

# Class weights computed only from the training labels.
counts = torch.bincount(y[train_ids], minlength=2).float()
class_weights = counts.sum() / (2.0 * counts.clamp(min=1))
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

def make_baseline_corrected():
    return BaselineGAT(3, hidden_channels=16, heads=2)

def make_ics_corrected():
    return GATWithICS(5, hidden_channels=16, heads=2)

def make_coherence_corrected():
    return CoherenceAwareGAT(
        in_channels=3,
        hidden_channels=16,
        heads=2,
        coherence_lambda=1.0,
        dropout=0.2
    )

configs = {
    "Baseline GAT": (make_baseline_corrected, 3, False),
    "GAT + ICS features": (make_ics_corrected, 5, False),
    "Coherence-Aware GAT": (make_coherence_corrected, 3, True),
}

def run_corrected_experiment(factory, n_features, uses_ics):
    x = x_scaled[:, :n_features].to(device)
    net = factory().to(device)
    optimizer = torch.optim.Adam(
        net.parameters(), lr=0.005, weight_decay=5e-4
    )

    best_val_ap = -1.0
    best_state = None
    best_epoch = 0
    patience = 8
    stale = 0

    for epoch in range(1, 41):
        net.train()
        optimizer.zero_grad()

        logits = (
            net(x, edge_index, ics)
            if uses_ics else net(x, edge_index)
        )

        # Correct two-class loss: both output logits are trained.
        loss = criterion(logits[train_ids], y[train_ids])
        loss.backward()
        optimizer.step()

        net.eval()
        with torch.no_grad():
            val_logits = (
                net(x, edge_index, ics)
                if uses_ics else net(x, edge_index)
            )
            val_probs = torch.softmax(val_logits[val_ids], dim=1)[:, 1]
            val_ap = average_precision_score(
                y[val_ids].cpu().numpy(),
                val_probs.cpu().numpy()
            )

        if val_ap > best_val_ap:
            best_val_ap = val_ap
            best_epoch = epoch
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in net.state_dict().items()
            }
            stale = 0
        else:
            stale += 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Epoch {epoch:02d} | loss={loss.item():.4f} "
                f"| validation PR-AUC={val_ap:.4f}"
            )

        if stale >= patience:
            print("Early stopping.")
            break

    net.load_state_dict(best_state)
    net.eval()

    with torch.no_grad():
        logits = (
            net(x, edge_index, ics)
            if uses_ics else net(x, edge_index)
        )

    val_prob = torch.softmax(logits[val_ids], dim=1)[:, 1].cpu().numpy()
    test_prob = torch.softmax(logits[test_ids], dim=1)[:, 1].cpu().numpy()
    val_true = y[val_ids].cpu().numpy()
    test_true = y[test_ids].cpu().numpy()

    # Threshold selection uses validation data only.
    p, r, thresholds = precision_recall_curve(val_true, val_prob)
    if len(thresholds):
        f1s = 2 * p[:-1] * r[:-1] / (p[:-1] + r[:-1] + 1e-12)
        threshold = float(thresholds[int(np.argmax(f1s))])
    else:
        threshold = 0.5

    pred = (test_prob >= threshold).astype(int)

    metrics = {
        "Best epoch": best_epoch,
        "Validation PR-AUC": best_val_ap,
        "Threshold": threshold,
        "Test accuracy": accuracy_score(test_true, pred),
        "Test precision": precision_score(test_true, pred, zero_division=0),
        "Test recall": recall_score(test_true, pred, zero_division=0),
        "Test F1": f1_score(test_true, pred, zero_division=0),
        "Test ROC-AUC": roc_auc_score(test_true, test_prob),
        "Test PR-AUC": average_precision_score(test_true, test_prob),
    }
    return net, metrics

corrected_trained_models = {}
corrected_rows = []

for name, (factory, n_features, uses_ics) in configs.items():
    print("\n" + "=" * 55 + "\n" + name + "\n" + "=" * 55)
    net, metrics = run_corrected_experiment(
        factory, n_features, uses_ics
    )
    corrected_trained_models[name] = net
    corrected_rows.append({"Model": name, **metrics})

corrected_comparison_df = pd.DataFrame(corrected_rows)
display(corrected_comparison_df.round(4))


Baseline GAT
Epoch 01 | loss=0.7450 | validation PR-AUC=0.1346
Epoch 05 | loss=0.7013 | validation PR-AUC=0.3470
Epoch 10 | loss=0.6574 | validation PR-AUC=0.7833
Epoch 15 | loss=0.6158 | validation PR-AUC=0.8049
Epoch 20 | loss=0.5582 | validation PR-AUC=0.8205
Epoch 25 | loss=0.5116 | validation PR-AUC=0.8152
Epoch 30 | loss=0.4513 | validation PR-AUC=0.8302
Epoch 35 | loss=0.4513 | validation PR-AUC=0.8439
Epoch 40 | loss=0.4192 | validation PR-AUC=0.8582

GAT + ICS features
Epoch 01 | loss=0.6945 | validation PR-AUC=0.6030
Epoch 05 | loss=0.6525 | validation PR-AUC=0.7253
Epoch 10 | loss=0.6201 | validation PR-AUC=0.7551
Epoch 15 | loss=0.5945 | validation PR-AUC=0.7623
Epoch 20 | loss=0.5575 | validation PR-AUC=0.7412
Early stopping.

Coherence-Aware GAT
Epoch 01 | loss=0.6825 | validation PR-AUC=0.6514
Epoch 05 | loss=0.6413 | validation PR-AUC=0.6574
Epoch 10 | loss=0.5893 | validation PR-AUC=0.7055
Epoch 15 | loss=0.5202 | validation PR-AUC=0.7692
Epoch 20 | loss=0.4809 | vali

,Model,Best epoch,Validation PR-AUC,Threshold,Test accuracy,Test precision,Test recall,Test F1,Test ROC-AUC,Test PR-AUC
0,Baseline GAT,39,0.8602,0.5101,0.9285,0.8526,0.8210,0.8365,0.9705,0.8657
1,GAT + ICS features,14,0.7636,0.5723,0.8913,0.7515,0.7654,0.7584,0.9451,0.8055
2,Coherence-Aware GAT,40,0.8479,0.5161,0.9120,0.7634,0.8765,0.8161,0.9618,0.8348


In [ ]:
import os

os.makedirs("/content/results", exist_ok=True)
os.makedirs("/content/models", exist_ok=True)

comparison_df.to_csv(
    "/content/results/coherence_aware_gat_comparison.csv",
    index=False
)

for name, trained_model in trained_models.items():
    filename = name.lower().replace(" ", "_").replace("+", "plus")
    torch.save(
        trained_model.state_dict(),
        f"/content/models/{filename}.pt"
    )

print("Comparison and model checkpoints saved in /content.")

Comparison and model checkpoints saved in /content.


In [ ]:

# Experiment: Test different coherence_lambda values
import copy
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score

# Reproducible experiment
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Use the existing graph, features, and splits
x3 = x_scaled[:, :3].detach().cpu().float()
edge_index = data.edge_index.detach().cpu().long()
ics = data.ics.detach().cpu().float().reshape(-1)
y = data.y.detach().cpu().long().reshape(-1)

train_ids = torch.as_tensor(train_idx).detach().cpu().long().reshape(-1)
val_ids = torch.as_tensor(val_idx).detach().cpu().long().reshape(-1)

# Handle class imbalance using training labels only
counts = torch.bincount(y[train_ids], minlength=2).float()
class_weights = counts.sum() / (2.0 * counts.clamp(min=1))
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))

# Lambda values to compare
lambda_values = [0, 0.25, 0.5, 1.0, 2.0]
lambda_results = []
lambda_models = {}

MAX_EPOCHS = 60
PATIENCE = 12

for lam in lambda_values:
    print(f"\n{'='*18} lambda = {lam} {'='*18}")

    # Reset random seed so each run starts comparably
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    model = CoherenceAwareGAT(
        in_channels=3,
        hidden_channels=16,
        heads=2,
        coherence_lambda=float(lam),
        dropout=0.2
    ).to(device)

    optimizer = torch.optim.Adam(
        model.parameters(), lr=0.005, weight_decay=5e-4
    )

    x_dev = x3.to(device)
    edge_dev = edge_index.to(device)
    ics_dev = ics.to(device)
    y_dev = y.to(device)
    train_dev = train_ids.to(device)
    val_dev = val_ids.to(device)

    best_val_pr_auc = -1.0
    best_epoch = 0
    best_state = None
    patience_counter = 0

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        optimizer.zero_grad()

        logits = model(x_dev, edge_dev, ics_dev)
        loss = criterion(logits[train_dev], y_dev[train_dev])
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(x_dev, edge_dev, ics_dev)
            val_probs = torch.softmax(val_logits[val_dev], dim=1)[:, 1]
            val_pr_auc = average_precision_score(
                y[val_ids].numpy(),
                val_probs.detach().cpu().numpy()
            )

        if val_pr_auc > best_val_pr_auc:
            best_val_pr_auc = val_pr_auc
            best_epoch = epoch
            best_state = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if epoch == 1 or epoch % 5 == 0:
            print(
                f"Epoch {epoch:02d} | loss={loss.item():.4f} | "
                f"validation PR-AUC={val_pr_auc:.4f}"
            )

        if patience_counter >= PATIENCE:
            print("Early stopping.")
            break

    # Restore the best validation checkpoint
    model.load_state_dict(best_state)
    lambda_models[lam] = copy.deepcopy(model).cpu()

    lambda_results.append({
        "coherence_lambda": lam,
        "best_epoch": best_epoch,
        "validation_PR_AUC": best_val_pr_auc
    })

lambda_comparison_df = pd.DataFrame(lambda_results).sort_values(
    "validation_PR_AUC", ascending=False
).reset_index(drop=True)

print("\nLAMBDA EXPERIMENT RESULTS")
display(lambda_comparison_df)

best_lambda = lambda_comparison_df.iloc[0]["coherence_lambda"]
print(f"\nBest lambda by validation PR-AUC: {best_lambda}")



================== lambda = 0 ==================
Epoch 01 | loss=0.7311 | validation PR-AUC=0.1426
Epoch 05 | loss=0.6958 | validation PR-AUC=0.2954
Epoch 10 | loss=0.6660 | validation PR-AUC=0.6659
Epoch 15 | loss=0.6255 | validation PR-AUC=0.6870
Epoch 20 | loss=0.5722 | validation PR-AUC=0.6957
Epoch 25 | loss=0.5221 | validation PR-AUC=0.6891
Epoch 30 | loss=0.4765 | validation PR-AUC=0.6834
Early stopping.

================== lambda = 0.25 ==================
Epoch 01 | loss=0.7327 | validation PR-AUC=0.1412
Epoch 05 | loss=0.6959 | validation PR-AUC=0.3014
Epoch 10 | loss=0.6645 | validation PR-AUC=0.6855
Epoch 15 | loss=0.6223 | validation PR-AUC=0.7085
Epoch 20 | loss=0.5669 | validation PR-AUC=0.7127
Epoch 25 | loss=0.5153 | validation PR-AUC=0.7050
Epoch 30 | loss=0.4695 | validation PR-AUC=0.6990
Early stopping.

================== lambda = 0.5 ==================
Epoch 01 | loss=0.7345 | validation PR-AUC=0.1400
Epoch 05 | loss=0.6960 | validation PR-AUC=0.3087
Epoch 10 | lo

,coherence_lambda,best_epoch,validation_PR_AUC
0,2.00,18,0.833338
1,1.00,19,0.772906
2,0.50,19,0.735531
3,0.25,19,0.715441
4,0.00,20,0.695665



Best lambda by validation PR-AUC: 2.0


In [ ]:

import numpy as np
import pandas as pd
import torch

from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix
)

# Retrieve the trained model for lambda = 2.0
best_lambda = 2.0
best_model = lambda_models[best_lambda].to(device)
best_model.eval()

x_eval = x3.to(device)
edges_eval = edge_index.to(device)
ics_eval = ics.to(device)

test_ids = torch.as_tensor(test_idx).detach().cpu().long().reshape(-1)

# Get probabilities for validation and test nodes
with torch.no_grad():
    logits = best_model(x_eval, edges_eval, ics_eval)
    probabilities = torch.softmax(logits, dim=1)[:, 1].cpu()

val_probs = probabilities[val_ids].numpy()
test_probs = probabilities[test_ids].numpy()

y_val = y[val_ids].numpy()
y_test = y[test_ids].numpy()

# Choose the threshold using validation data ONLY
precisions, recalls, thresholds = precision_recall_curve(
    y_val, val_probs
)

f1_values = (
    2 * precisions[:-1] * recalls[:-1]
    / (precisions[:-1] + recalls[:-1] + 1e-12)
)

best_threshold = float(thresholds[np.argmax(f1_values)])

# Evaluate test data once at the selected threshold
test_predictions = (test_probs >= best_threshold).astype(int)

results = {
    "Model": "Coherence-Aware GAT (lambda=2.0)",
    "Validation PR-AUC": average_precision_score(y_val, val_probs),
    "Selected threshold": best_threshold,
    "Test accuracy": accuracy_score(y_test, test_predictions),
    "Test precision": precision_score(
        y_test, test_predictions, zero_division=0
    ),
    "Test recall": recall_score(
        y_test, test_predictions, zero_division=0
    ),
    "Test F1": f1_score(
        y_test, test_predictions, zero_division=0
    ),
    "Test ROC-AUC": roc_auc_score(y_test, test_probs),
    "Test PR-AUC": average_precision_score(y_test, test_probs)
}

display(pd.DataFrame([results]).round(4))

print("Test confusion matrix:")
print(confusion_matrix(y_test, test_predictions))


,Model,Validation PR-AUC,Selected threshold,Test accuracy,Test precision,Test recall,Test F1,Test ROC-AUC,Test PR-AUC
0,Coherence-Aware GAT (lambda=2.0),0.8333,0.5416,0.9078,0.7684,0.8395,0.8024,0.9552,0.835


Test confusion matrix:
[[524  41]
 [ 26 136]]


In [ ]:

import torch
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score

model = lambda_models[2.0].to(device)
model.eval()

with torch.no_grad():
    logits = model(
        x3.to(device),
        edge_index.to(device),
        ics.to(device)
    )
    probs = torch.softmax(logits, dim=1).cpu().numpy()

y_check = data.y.detach().cpu().numpy().reshape(-1).astype(int)
test_check = torch.as_tensor(test_idx).cpu().numpy().reshape(-1)

print("Logits shape:", tuple(logits.shape))
print("Label values/counts:", np.unique(y_check, return_counts=True))
print("First 10 logits:\n", logits[:10].cpu().numpy())
print("First 10 class probabilities:\n", probs[:10])
print("Mean class-0 probability:", probs[:, 0].mean())
print("Mean class-1 probability:", probs[:, 1].mean())
print("Test actual fraud rate:", y_check[test_check].mean())
print(
    "Test ROC-AUC:",
    roc_auc_score(y_check[test_check], probs[test_check, 1])
)
print(
    "Test PR-AUC:",
    average_precision_score(y_check[test_check], probs[test_check, 1])
)


Logits shape: (4906, 2)
Label values/counts: (array([0, 1]), array([3774, 1132]))
First 10 logits:
 [[ 0.6326247  -0.39475414]
 [-0.00327463  0.13571946]
 [ 0.06395387  0.07839268]
 [-0.0327254   0.15403451]
 [-0.64843094  0.6045987 ]
 [-0.26978332  0.3275956 ]
 [-0.3586136   0.423289  ]
 [ 0.22320496 -0.0118757 ]
 [ 0.46581584 -0.22706965]
 [-0.09768783  0.15277554]]
First 10 class probabilities:
 [[0.7364074  0.2635926 ]
 [0.46530733 0.5346927 ]
 [0.49639037 0.50360966]
 [0.45344526 0.54655474]
 [0.22217615 0.77782387]
 [0.3549436  0.6450564 ]
 [0.31390998 0.68609005]
 [0.558501   0.441499  ]
 [0.6666085  0.33339146]
 [0.43770945 0.56229055]]
Mean class-0 probability: 0.5355949
Mean class-1 probability: 0.46440512
Test actual fraud rate: 0.22283356258596973
Test ROC-AUC: 0.9551513165082486
Test PR-AUC: 0.8350287369193216


In [58]:
results = """
MAJOR PROJECT: Synthetic Identity Fraud Detection
Version: 1

ICS-ONLY:
ROC-AUC: 0.8468
PR-AUC: 0.7732
Precision@100: 1.0000
Precision@Top 1%: 1.0000
Precision@Top 2%: 1.0000
Precision@Top 5%: 1.0000

BASELINE GAT:
Test Accuracy: 0.9285
Precision: 0.8526
Recall: 0.8210
F1: 0.8365
ROC-AUC: 0.9705
PR-AUC: 0.8657

GAT + ICS FEATURES:
Test Accuracy: 0.8913
Precision: 0.7515
Recall: 0.7654
F1: 0.7584
ROC-AUC: 0.9451
PR-AUC: 0.8055

COHERENCE-AWARE GAT (LAMBDA=2):
Test Accuracy: 0.9078
Precision: 0.7684
Recall: 0.8395
F1: 0.8024
ROC-AUC: 0.9552
PR-AUC: 0.8350

Confusion Matrix:
[[524, 41],
 [26, 136]]

Note: Results are from the current synthetic dataset.
"""

with open("/content/Major_Project_Results_V1.txt", "w") as f:
    f.write(results)

print("Results file saved successfully!")

Results file saved successfully!


In [59]:
from google.colab import files
files.download("/content/Major_Project_Results_V1.txt")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [60]:
import torch

torch.save(
    lambda_models[2.0].state_dict(),
    "/content/coherence_gat_lambda2_v1.pth"
)

In [64]:
import os
import numpy as np
import pandas as pd
import torch
from google.colab import drive

# Google Drive mount
drive.mount("/content/drive")

SAVE_DIR = "/content/drive/MyDrive/Major_Project_V1"
os.makedirs(SAVE_DIR, exist_ok=True)

# Existing trained model — retraining nahi karni
model = lambda_models[2.0]
model.eval()

with torch.no_grad():
    logits = model(x3, edge_index, ics)
    probabilities = torch.softmax(logits, dim=1)[:, 1]

risk_prob = probabilities.detach().cpu().numpy()
labels = data.y.detach().cpu().numpy()
ics_values = data.ics.detach().cpu().numpy()

# Original graph features:
# [out_degree, in_degree, total_degree, ICS, ICS risk]
features = data.x.detach().cpu().numpy()

# V1 predictions: threshold previously selected on validation set
threshold = 0.5416

results = pd.DataFrame({
    "node_id": [f"NODE_{i:05d}" for i in range(len(labels))],
    "out_degree": features[:, 0],
    "in_degree": features[:, 1],
    "total_degree": features[:, 2],
    "ics": ics_values,
    "ics_risk": 1 - ics_values,
    "fraud_probability": risk_prob,
    "predicted_fraud": (risk_prob >= threshold).astype(int),
    "actual_label": labels
})

results.to_csv(
    f"{SAVE_DIR}/graph_risk_results.csv",
    index=False
)

print("Export complete!")
print("Rows:", len(results))
print("Saved:", f"{SAVE_DIR}/graph_risk_results.csv")
print(results.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Export complete!
Rows: 4906
Saved: /content/drive/MyDrive/Major_Project_V1/graph_risk_results.csv
      node_id  out_degree  in_degree  total_degree       ics  ics_risk  \
0  NODE_00000        13.0        8.0          21.0  1.000000  0.000000   
1  NODE_00001         9.0       17.0          26.0  1.000000  0.000000   
2  NODE_00002        11.0        4.0          15.0  1.000000  0.000000   
3  NODE_00003        13.0       27.0          40.0  0.193487  0.806513   
4  NODE_00004        15.0       10.0          25.0  0.193487  0.806513   

   fraud_probability  predicted_fraud  actual_label  
0           0.263593                0             0  
1           0.534693                0             1  
2           0.503610                0             0  
3           0.546555                1             1  
4           0.777824                1             1  


In [63]:
from google.colab import drive

drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [65]:

from pathlib import Path

project_dir = Path("/content/drive/MyDrive/Major_Project_V1")
project_dir.mkdir(parents=True, exist_ok=True)

app_code = r'''
import streamlit as st
import pandas as pd
from pathlib import Path

st.set_page_config(
    page_title="Synthetic Identity Fraud Detection",
    page_icon="🛡️",
    layout="wide"
)

DATA_PATH = Path(__file__).parent / "graph_risk_results.csv"
THRESHOLD = 0.5416

st.title("🛡️ Synthetic Identity Fraud Detection")
st.caption("Digital Lending | Graph-based Risk Analysis")

if not DATA_PATH.exists():
    st.error("Dataset file missing. Keep graph_risk_results.csv beside app.py.")
    st.stop()

df = pd.read_csv(DATA_PATH)

required = {
    "node_id", "out_degree", "in_degree", "total_degree",
    "ics", "ics_risk", "fraud_probability", "predicted_fraud"
}
if not required.issubset(df.columns):
    st.error("Dataset columns do not match the expected format.")
    st.stop()

df["risk_level"] = pd.cut(
    df["fraud_probability"],
    bins=[-0.001, 0.30, THRESHOLD, 1.001],
    labels=["Low", "Medium", "High"],
    right=False
)

page = st.sidebar.radio(
    "Navigate",
    ["Dashboard", "Risk Explorer", "Model Information"]
)

st.sidebar.info(
    "Research prototype using a synthetic graph dataset. "
    "Not validated for real lending decisions."
)

if page == "Dashboard":
    st.subheader("Graph Risk Overview")

    c1, c2, c3, c4 = st.columns(4)
    c1.metric("Graph Nodes", f"{len(df):,}")
    c2.metric("High Risk Nodes", int((df["predicted_fraud"] == 1).sum()))
    c3.metric("Average ICS", f'{df["ics"].mean():.3f}')
    c4.metric("Average Risk Score", f'{df["fraud_probability"].mean():.3f}')

    st.subheader("Predicted Risk Distribution")
    risk_counts = (
        df["risk_level"].value_counts()
        .reindex(["Low", "Medium", "High"], fill_value=0)
    )
    st.bar_chart(risk_counts)

    st.subheader("Highest-Risk Graph Nodes")
    display = df.sort_values(
        "fraud_probability", ascending=False
    ).head(15)

    # Do not display actual labels in the user-facing dashboard.
    st.dataframe(
        display[[
            "node_id", "fraud_probability", "risk_level",
            "ics", "ics_risk", "in_degree", "out_degree"
        ]],
        use_container_width=True,
        hide_index=True
    )

elif page == "Risk Explorer":
    st.subheader("Explore a Synthetic Graph Node")

    search = st.text_input(
        "Search node ID", placeholder="e.g. NODE_00003"
    ).strip().upper()

    filtered = df
    level = st.selectbox("Filter risk level", ["All", "Low", "Medium", "High"])

    if level != "All":
        filtered = filtered[filtered["risk_level"].astype(str) == level]

    if search:
        filtered = filtered[filtered["node_id"].str.upper().str.contains(search, regex=False)]

    st.write(f"Matching nodes: {len(filtered)}")

    if len(filtered):
        chosen = st.selectbox("Select node", filtered["node_id"].tolist())
        row = df.loc[df["node_id"] == chosen].iloc[0]

        a, b, c = st.columns(3)
        a.metric("Fraud Probability", f'{row["fraud_probability"]:.1%}')
        b.metric("Predicted Risk", str(row["risk_level"]))
        c.metric("Identity Coherence Score", f'{row["ics"]:.3f}')

        d, e, f = st.columns(3)
        d.metric("Incoming Connections", int(row["in_degree"]))
        e.metric("Outgoing Connections", int(row["out_degree"]))
        f.metric("Total Degree", int(row["total_degree"]))

        st.progress(float(row["fraud_probability"]))
        st.caption(
            "Probability is the model's score, not a verified probability "
            "of real-world fraud. Node-level ICS is inherited from its applicant."
        )
    else:
        st.warning("No matching nodes found.")

elif page == "Model Information":
    st.subheader("Methodology")

    st.markdown("""
    This prototype explores precomputed outputs from a synthetic,
    transaction-linked graph.

    **Selected model:** Coherence-Aware Graph Attention Network (GAT),
    with lambda = 2.0.

    **Node features:** outgoing degree, incoming degree, total degree,
    Identity Coherence Score (ICS), and ICS-derived risk.

    **Prediction threshold:** 0.5416, selected using the validation set
    in the existing experiment.

    **Important limitation:** This app explores existing graph nodes.
    It does not create a new applicant graph or score arbitrary new
    applicants live. The dataset is synthetic and includes intentionally
    encoded fraud-related patterns.
    """)

    st.subheader("Existing Test Results")
    metrics = pd.DataFrame({
        "Metric": ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "PR-AUC"],
        "Score": [0.9078, 0.7684, 0.8395, 0.8024, 0.9552, 0.8350]
    })
    st.dataframe(metrics, hide_index=True, use_container_width=True)

    st.warning(
        "These are the previously recorded test results for lambda = 2.0. "
        "They are reported for the existing experiment, not recomputed by this app."
    )
'''

(project_dir / "app.py").write_text(app_code, encoding="utf-8")

requirements = """streamlit
pandas
"""
(project_dir / "requirements.txt").write_text(requirements, encoding="utf-8")

print("Created:")
print(project_dir / "app.py")
print(project_dir / "requirements.txt")
print("CSV exists:", (project_dir / "graph_risk_results.csv").exists())


Created:
/content/drive/MyDrive/Major_Project_V1/app.py
/content/drive/MyDrive/Major_Project_V1/requirements.txt
CSV exists: True


In [66]:

from google.colab import files

files.download("/content/drive/MyDrive/Major_Project_V1/app.py")
files.download("/content/drive/MyDrive/Major_Project_V1/requirements.txt")
files.download("/content/drive/MyDrive/Major_Project_V1/graph_risk_results.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>